# InsertionSort (Java)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap13_correttezza/InsertionSort_java.ipynb)

Capitolo 13 — Correttezza. Notebook eseguibile di `InsertionSort`: l'algoritmo in Java, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/analisi/13-correttezza/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap13_correttezza/InsertionSort_python.ipynb) e [C++](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap13_correttezza/InsertionSort_cpp.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap13_correttezza_java.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. La prima cella verifica che il JDK ci sia e, se manca, lo installa. Ogni programma Java è scritto su un file dalla cella `%%writefile`, poi compilato con `javac` ed eseguito con `java` nella cella successiva.

**Indici.** Nello pseudocodice gli array sono indicizzati da 1; nel codice Java da 0: l'elemento `a[i]` dello pseudocodice è `a[i-1]` nel codice (e un ciclo `while i <= n` da 1 diventa `while i < n` da 0). Le tabelle stampate usano gli indici dello pseudocodice.

## Preparazione

Su Colab il JDK di solito c'è già: la cella qui sotto lo verifica e, se manca, lo installa.

In [1]:
!javac -version || (apt-get -qq update && apt-get -qq install -y default-jdk-headless)

javac 25.0.4.1


## Ordinamento per inserimento: InsertionSort

**Input:** un array $a[1..n]$ di $n \ge 1$ numeri naturali. **Output:** l'array ordinato in modo non decrescente. **Invariante esterno:** $a[1..i-1]$ contiene gli elementi originali di $a[1..i-1]$ ordinati e $a[i..n]$ è uguale all'originale. **Invariante interno** (I1)–(I3): $a[1..j] = b[1..j]$, $a[j+2..i] = b[j+1..i-1]$ e questi elementi sono $> key$.

Pseudocodice: sezione 13.3 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/analisi/13-correttezza/).

In [2]:
%%writefile InsertionSort.java
public class InsertionSort
{
// "[a[from], ..., a[to-1]]"
static String fmt(int[] a, int from, int to)
{
    StringBuilder s = new StringBuilder("[");
    for (int i = from; i < to; i++)
    {
        if (i > from) s.append(", ");
        s.append(a[i]);
    }
    return s.append("]").toString();
}

static boolean equal(int[] x, int xs, int[] y, int ys, int len)     // x[xs..xs+len-1] == y[ys..ys+len-1]
{
    for (int k = 0; k < len; k++)
        if (x[xs + k] != y[ys + k]) return false;
    return true;
}

    // L'array con una barra dopo la parte ordinata a[1..i-1]
    static String split(int[] a, int i)
    {
        String s = fmt(a, 0, i);
        String t = fmt(a, i, a.length);
        return s.substring(0, s.length() - 1) + " | " + t.substring(1);
    }

    // Invariante esterno: a[1..i-1] = elementi originali di a[1..i-1] ordinati, a[i..n] = originale
    static void printOuter(int[] a, int[] orig, int i)
    {
        int n = a.length;
        int[] s = java.util.Arrays.copyOf(orig, i);
        java.util.Arrays.sort(s);
        boolean ok = equal(a, 0, s, 0, i) && equal(a, i, orig, i, n - i);
        System.out.println(String.format("%2d | %-26s | %-19s | %s", i + 1, split(a, i), fmt(orig, 0, i), ok ? "ok" : "FALSA"));
    }

    // Invariante interno (I1)-(I3); b = a a inizio iterazione
    static void printInner(int[] a, int[] b, int i, int j, int key)
    {
        int ip = i + 1, jp = j + 1;                // indici dello pseudocodice
        boolean ok = equal(a, 0, b, 0, jp) && equal(a, jp + 1, b, jp, ip - jp - 1);
        for (int k = jp + 1; k < ip; k++)
            if (a[k] <= key) ok = false;
        String cond = "j = 0: false";
        if (j >= 0)
        {
            boolean t = a[j] > key;
            cond = "a[" + jp + "] = " + a[j] + (t ? " > " : " <= ") + key + ": " + (t ? "true" : "false");
        }
        System.out.println(String.format("%d | %-15s | %-12s | %-9s | %-20s | %s", jp, fmt(a, 0, a.length), fmt(a, 0, jp), fmt(a, jp + 1, ip), cond, ok ? "ok" : "FALSA"));
    }

    // InsertionSort(a, n)
    static void insertionSort(int[] a, boolean showOuter, int traceInnerAt)
    {
        int n = a.length;
        int[] orig = a.clone();                    // copia dell'input: serve solo a verificare l'invariante
        int i = 1;                                 // i := 2
        if (showOuter) printOuter(a, orig, i);
        while (i < n)                              // i <= n
        {
            int[] b = a.clone();                   // contenuto di a a inizio iterazione (solo per l'invariante interno)
            int key = a[i];
            int j = i - 1;
            boolean tr = (i + 1 == traceInnerAt);
            if (tr) printInner(a, b, i, j, key);
            while (j >= 0 && a[j] > key)           // j >= 1 and a[j] > key
            {
                a[j + 1] = a[j];
                j = j - 1;
                if (tr) printInner(a, b, i, j, key);
            }
            a[j + 1] = key;
            i = i + 1;
            if (showOuter) printOuter(a, orig, i);
        }
    }

    public static void main(String[] args)
    {
        // Il ciclo esterno: a[1..5] = (5, 2, 4, 1, 3), esempio delle dispense
        int[] a = {5, 2, 4, 1, 3};
        int n = a.length;
        System.out.printf("a = %s\n", fmt(a, 0, n));
        System.out.printf(" i | a (ordinato | resto)       | a[1..i-1] originale | invariante\n");
        insertionSort(a, true, 0);
        System.out.printf("ordinato: %s\n", fmt(a, 0, n));

        // Il ciclo interno all'iterazione i = 5, con key = 3 (a = (1, 2, 4, 5, 3) all'inizio)
        System.out.printf("\n");
        int[] c = {5, 2, 4, 1, 3};
        System.out.printf("j | a               | a[1..j]      | a[j+2..i] | condizione           | invariante\n");
        insertionSort(c, false, 5);
        System.out.printf("dopo la riga 10 (a[j+1] = key): a = %s\n", fmt(c, 0, n));
    }
}

Writing InsertionSort.java


In [3]:
!javac InsertionSort.java && java InsertionSort

a = [5, 2, 4, 1, 3]
 i | a (ordinato | resto)       | a[1..i-1] originale | invariante
 2 | [5 | 2, 4, 1, 3]           | [5]                 | ok
 3 | [2, 5 | 4, 1, 3]           | [5, 2]              | ok
 4 | [2, 4, 5 | 1, 3]           | [5, 2, 4]           | ok
 5 | [1, 2, 4, 5 | 3]           | [5, 2, 4, 1]        | ok
 6 | [1, 2, 3, 4, 5 | ]         | [5, 2, 4, 1, 3]     | ok
ordinato: [1, 2, 3, 4, 5]

j | a               | a[1..j]      | a[j+2..i] | condizione           | invariante
4 | [1, 2, 4, 5, 3] | [1, 2, 4, 5] | []        | a[4] = 5 > 3: true   | ok
3 | [1, 2, 4, 5, 5] | [1, 2, 4]    | [5]       | a[3] = 4 > 3: true   | ok
2 | [1, 2, 4, 4, 5] | [1, 2]       | [4, 5]    | a[2] = 2 <= 3: false | ok
dopo la riga 10 (a[j+1] = key): a = [1, 2, 3, 4, 5]


**Prova tu.** Cambia l'array: ogni riga della tabella esterna deve restare `ok`.

In [4]:
%%writefile ProvaInsertionSort.java
public class ProvaInsertionSort
{
    public static void main(String[] args)
    {
        int[] a = {6, 3, 8, 1, 9, 2};      // <-- cambia l'array
        System.out.printf(" i | a (ordinato | resto)       | a[1..i-1] originale | invariante\n");
        InsertionSort.insertionSort(a, true, 0);
    }
}

Writing ProvaInsertionSort.java


In [5]:
!javac ProvaInsertionSort.java && java ProvaInsertionSort

 i | a (ordinato | resto)       | a[1..i-1] originale | invariante
 2 | [6 | 3, 8, 1, 9, 2]        | [6]                 | ok
 3 | [3, 6 | 8, 1, 9, 2]        | [6, 3]              | ok
 4 | [3, 6, 8 | 1, 9, 2]        | [6, 3, 8]           | ok
 5 | [1, 3, 6, 8 | 9, 2]        | [6, 3, 8, 1]        | ok
 6 | [1, 3, 6, 8, 9 | 2]        | [6, 3, 8, 1, 9]     | ok
 7 | [1, 2, 3, 6, 8, 9 | ]      | [6, 3, 8, 1, 9, 2]  | ok


---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).